# 🧠 AccessLens — Advanced ViT Training Notebook (v3)

**What this notebook does:**
1. Upload project zip
2. Install all dependencies
3. Download **large real-world dataset** from HuggingFace (10,000+ samples from 4 sources)
4. Apply weak supervision labeling (OpenCV heuristics)
5. Generate 1,000 targeted synthetic samples
6. Train **Vision Transformer ViT-B/16** (86M params) — 30 epochs
7. Evaluate with per-class F1, Hamming Loss
8. Deploy with public URL

> ⚡ **Runtime**: Set to **T4 GPU** → Runtime → Change runtime type → T4 GPU

---
**v3 Changes**: Larger dataset (10K+ real + 1K synthetic = 11K+), additional HF sources, improved training

## Step 1: Upload Project

In [ ]:
from google.colab import files
import zipfile, os

print('📂 Upload your accesslens.zip...')
uploaded = files.upload()
for f in uploaded:
    with zipfile.ZipFile(f,'r') as z:
        z.extractall('/content/accesslens')
os.chdir('/content/accesslens')
!ls -la
print('✅ Project uploaded!')

## Step 2: Install Dependencies

In [ ]:
%%time
print('📦 System deps...')
!sudo apt-get install -y -qq chromium-browser chromium-chromedriver libgl1-mesa-glx > /dev/null 2>&1

print('📦 Python packages...')
!pip install -q fastapi uvicorn beautifulsoup4 selenium webdriver-manager httpx \
    pydantic python-multipart lxml Pillow cssutils pydantic-settings
!pip install -q datasets huggingface-hub   # Real dataset download
!pip install -q opencv-python-headless     # Weak supervision labeling
!pip install -q timm                        # ViT model support
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

print('✅ All dependencies installed!')
import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

## Step 2.5: Google Drive Sync (Skip Training if Model Exists)

We'll check if the trained model already exists in your Google Drive. If it does, we can skip the dataset generation and the 30-epoch training phase entirely!

In [ ]:
from google.colab import drive
import os, shutil

print('Mounting Google Drive...')
drive.mount('/content/drive')

model_save_path = '/content/drive/MyDrive/accesslens_vit_b16.pth'
local_model_path = 'dataset/accessibility_model.pth'
os.makedirs('dataset', exist_ok=True)

SKIP_TRAINING = False
if os.path.exists(model_save_path):
    print(f'\n✅ Found pre-trained model: {model_save_path}')
    print('Copying model to local environment...')
    shutil.copy(model_save_path, local_model_path)
    SKIP_TRAINING = True
    print('✅ Dataset generation and training will be SKIPPED.')
else:
    print('\n🚀 No pre-trained model found. Proceeding with full data generation and training pipeline.')

## Step 3: Download Large Real-World Dataset (10,000+ samples)

Downloads **real web screenshots** from multiple HuggingFace sources:
- `osunlp/Multimodal-Mind2Web` — real browser screenshots (4,000 samples)
- `biglab/webui-7k` — real web UI images (3,000 samples)
- `ZhihaoNan/AtomBlock-WebUI` — additional web screenshots (2,000 samples)
- `nickmuchi/financial-app-ui` — financial app UIs (1,000 samples)

Then generates **1,000** targeted synthetic samples for hard-to-detect classes.

**Total: ~11,000+ labeled samples** (2x the previous dataset)

In [ ]:
%%time
if not SKIP_TRAINING:
    import sys, logging
    sys.path.insert(0,'/content/accesslens')
    logging.basicConfig(level=logging.INFO, format='%(message)s')
    
    from backend.ml.dataset_generator import DatasetGenerator
    
    print('🌐 Downloading large real web screenshot dataset from HuggingFace...')
    print('📊 Applying weak supervision labels (OpenCV heuristics)...')
    print('🎨 Generating 1,000 targeted synthetic samples...')
    print()
    
    gen = DatasetGenerator(output_dir='dataset', num_synthetic=1000, num_real=10000)
    gen.generate()
    
    print(f'\n✅ Dataset ready: {len(gen.metadata)} total samples')
else:
    print('⏭️ Skipping Dataset Generation (Model already loaded from Drive)')

In [ ]:
if not SKIP_TRAINING:
    # Verify dataset distribution
    import json
    import matplotlib.pyplot as plt
    from collections import Counter

    with open('dataset/metadata.json') as f:
        meta = json.load(f)

    print(f'Total samples: {meta["num_samples"]}')
    print(f'Sources: {meta.get("sources",{})}')

    # Class distribution
    label_counts = Counter()
    for s in meta['samples']:
        for n in s['label_names']:
            label_counts[n] += 1
    label_counts['clean'] = sum(1 for s in meta['samples'] if not s['label_names'])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))

    # Violation distribution
    ax1.barh(list(label_counts.keys()), list(label_counts.values()),
             color=['#ef4444','#f59e0b','#10b981','#3b82f6','#8b5cf6','#ec4899','#6b7280'])
    ax1.set_title('Violation Class Distribution')
    ax1.set_xlabel('Count')

    # Source breakdown
    sources = meta.get('sources',{})
    ax2.pie(sources.values(), labels=sources.keys(), autopct='%1.1f%%', startangle=90)
    ax2.set_title('Data Sources')

    plt.tight_layout()
    plt.show()
    print(f'\nClass counts: {dict(label_counts)}')
else:
    print('⏭️ Skipping Dataset Verification')

## Step 4: Train Vision Transformer ViT-B/16

**Architecture**: ViT-B/16 (86M params, pretrained on ImageNet-1k)

**Training Strategy**:
- Phase 1 (Epochs 1-5): Frozen backbone → head warmup
- Phase 2 (Epochs 6-30): Unfrozen, layer-wise LR decay

**Techniques**: MixUp, CutMix, RandAugment, Random Erasing, OneCycleLR, AMP, EMA, Weighted Sampling, Early Stopping

**Dataset**: 11,000+ samples (10K real + 1K synthetic)

In [ ]:
from backend.ml.model import get_model, count_parameters
import torch

# Show model architecture
model = get_model(variant='vit_b16', pretrained=True, freeze_backbone=True)
params = count_parameters(model)
print(f'Model: {getattr(model,"variant","?")} (Vision Transformer B/16)')
print(f'Total parameters:     {params["total_millions"]:>10}')
print(f'Trainable (Phase 1):  {params["trainable_millions"]:>10}  (head only)')
print(f'Device: {"T4 GPU" if torch.cuda.is_available() else "CPU (install GPU runtime!)"}')

In [ ]:
%%time
if not SKIP_TRAINING:
    from backend.ml.train import train_model
    import shutil
    
    history = train_model(
        dataset_dir='dataset',
        epochs=30,
        batch_size=32,
        grad_accum_steps=2,
        phase1_epochs=5,
        learning_rate=3e-4,
        backbone_lr_mult=0.1,
        weight_decay=1e-4,
        label_smoothing=0.1,
        mixup_alpha=0.4,
        cutmix_alpha=1.0,
        use_mixup=True,
        use_ema=True,
        early_stop_patience=7,
        model_variant='vit_b16',
    )
    print(f'\n✅ Training Complete!')
    print(f'Best Epoch: {history["best_epoch"]} | Best Val F1: {history["best_val_f1"]:.4f}')
    print('Saving model to Google Drive...')
    shutil.copy('dataset/accessibility_model.pth', '/content/drive/MyDrive/accesslens_vit_b16.pth')
    print('✅ Saved!')
else:
    print('⏭️ Skipping Training (Model already loaded from Drive)')

In [ ]:
if not SKIP_TRAINING:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss
    axes[0].plot(history['train_loss'], label='Train', linewidth=2)
    axes[0].plot(history['val_loss'],   label='Val',   linewidth=2)
    axes[0].axvline(x=4, color='orange', linestyle='--', alpha=0.7, label='Phase 2 start')
    axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # F1
    axes[1].plot(history['train_f1'], label='Train F1', linewidth=2)
    axes[1].plot(history['val_f1'],   label='Val F1',   linewidth=2)
    axes[1].axhline(y=history['best_val_f1'], color='r', linestyle='--',
                   label=f'Best: {history["best_val_f1"]:.3f}')
    axes[1].set_title('Macro F1 Score'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    # Per-class F1
    if history['per_class_f1']:
        best = history['per_class_f1'][history['best_epoch']-1]
        cls_names = list(best.keys())
        cls_f1s   = list(best.values())
        colors = ['#ef4444','#f59e0b','#10b981','#3b82f6','#8b5cf6','#ec4899']
        bars = axes[2].bar(range(len(cls_names)), cls_f1s, color=colors)
        axes[2].set_xticks(range(len(cls_names)))
        axes[2].set_xticklabels(cls_names, rotation=35, ha='right', fontsize=9)
        axes[2].set_ylim(0,1); axes[2].set_title('Per-Class F1 (Best Epoch)')
        for bar, v in zip(bars, cls_f1s):
            axes[2].text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.2f}', ha='center', fontsize=9)

    plt.suptitle(f'AccessibilityViT Training — 11K+ Dataset — {getattr(model,"variant","ViT-B/16")}', fontsize=14)
    plt.tight_layout(); plt.show()
else:
    print('⏭️ Skipping Training Plots')

## Step 5: Evaluate on Test Set

In [ ]:
if not SKIP_TRAINING:
    from backend.ml.inference import AccessibilityInference
    from backend.ml.train import AccessibilityDataset, get_val_transform, compute_metrics
    from torch.utils.data import DataLoader
    import torch, json

    with open('dataset/metadata.json') as f: meta=json.load(f)
    with open('dataset/splits.json')   as f: splits=json.load(f)

    test_s = [meta['samples'][i] for i in splits['test']]
    test_ds = AccessibilityDataset(test_s, 'dataset/images', get_val_transform())
    test_dl = DataLoader(test_ds, batch_size=32, shuffle=False)

    inf = AccessibilityInference(model_path='dataset/accessibility_model.pth')
    print(f'Model variant: {inf.model.variant if inf.model else "?"}')

    inf.model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for imgs, lbls in test_dl:
            imgs = imgs.to(inf.device)
            all_p.append(inf.model(imgs).cpu())
            all_t.append(lbls)

    m = compute_metrics(torch.cat(all_p), torch.cat(all_t))
    print(f'\n=== TEST SET RESULTS ===')
    print(f'Macro F1:      {m["macro_f1"]:.4f}')
    print(f'Hamming Loss:  {m["hamming_loss"]:.4f}')
    print(f'Exact Match:   {m["exact_match"]:.4f}')
    print(f'\nPer-Class F1:')
    from backend.ml.model import VIOLATION_CLASSES
    for cls in VIOLATION_CLASSES:
        f1 = m[f'{cls}_f1']; bar='█'*int(f1*25)
        print(f'  {cls:20s}: {f1:.4f}  {bar}')
else:
    print('⏭️ Skipping Evaluation (Requires full dataset which was skipped)')

## Step 6: Start Web Server + Public URL

In [ ]:
import subprocess, time
proc = subprocess.Popen(
    ['python','-m','uvicorn','backend.main:app','--host','0.0.0.0','--port','8000'],
    cwd='/content/accesslens'
)
time.sleep(4)
print('✅ Server started on port 8000')

In [ ]:
import time
print('Starting Cloudflare Tunnel...')
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8000 > cloudflared.log 2>&1 &

time.sleep(5)
print('\n🌐 AccessLens is LIVE at:')
!grep -o 'https://.*\.trycloudflare.com' cloudflared.log
print('\nClick the link above to open your dashboard!')